In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [3]:
# Running the setup

!pip install faiss-cpu #install FAISS

import pandas as pd 
import numpy as np 
import faiss 
from sentence_transformers import SentenceTransformer, CrossEncoder 
from transformers import AutoTokenizer, pipeline 
from sklearn.feature_extraction.text import TfidfVectorizer 
from sklearn.metrics.pairwise import cosine_similarity 

train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv') 

print("Creating nowledge base")

kb = [] 

for idx, row in train.iterrows(): 
    correct_letter = row['answer'] 
    kb.append(str(row[correct_letter])) 

print("Loading embedding model and creating index") 

model = SentenceTransformer('all-MiniLM-L6-v2') 
kb_embeddings = model.encode(kb, show_progress_bar=False) 
index = faiss.IndexFlatL2(kb_embeddings.shape[1]) 
index.add(kb_embeddings)

print("Knowledge base successfully created")

Creating nowledge base
Loading embedding model and creating index


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Knowledge base successfully created


**Zero-shot classifier for Q1, Q2, Q6**

In [4]:
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli") 

row_150 = train.iloc[150] 
prompt_150 = str(row_150['prompt']) 

labels_150 = [str(row_150['A']), str(row_150['B']), str(row_150['C']), str(row_150['D']), str(row_150['E'])] 

ans_150 = str(row_150[row_150['answer']])

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Q1. Run the zero-shot classifier on facebook/bart-large-mnli on prompt for the row index 150. Pass the 5 options (A-E) candidate_labels. What is the predicted probability score assigned to the ground-truth correct option (option in the answer column)? (Round to 3 decimal points)

In [5]:
# Run zero-shot classification
result = zs(
    sequences=prompt_150,
    candidate_labels=labels_150,
    multi_label=False
)

# Show all scores
for label, score in zip(result["labels"], result["scores"]):
    print(f"{label[:80]}... -> {score:.6f}")

# Find the score of the ground-truth answer
ground_truth_score = dict(zip(result["labels"], result["scores"]))[ans_150]

print("\nGround Truth Answer:")
print(ans_150)

print(f"\nPredicted probability score: {ground_truth_score:.3f}")

The butterfly effect is the phenomenon that a small change in the initial condit... -> 0.384420
The butterfly effect is the phenomenon that a large change in the initial condit... -> 0.378677
The butterfly effect is the phenomenon that a small change in the initial condit... -> 0.092697
The butterfly effect is the phenomenon that a small change in the initial condit... -> 0.078618
The butterfly effect is the phenomenon that a large change in the initial condit... -> 0.065588

Ground Truth Answer:
The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."

Predicted probability score: 0.384


Q2. Embed the prompt for row index 150 using all-MiniLM-L6-v2. Query your FAISS index to retrieve the top k=10 most similar documents. At what exact rank (1 through 10) did FAISS place the true correct document (which is the document originally located at index 150 in the KB)?

In [6]:
# Embed the prompt
query_embedding = model.encode([prompt_150])

# Search top-10 nearest neighbours
k = 10
distances, indices = index.search(query_embedding, k)

print("Top-10 retrieved document indices:")
print(indices[0])

# Find the rank of the true document (KB index 150)
true_doc_index = 150

if true_doc_index in indices[0]:
    rank = list(indices[0]).index(true_doc_index) + 1
    print(f"\nTrue document found at Rank: {rank}")
else:
    print("\nTrue document NOT found in the top-10 retrieved documents.")

Top-10 retrieved document indices:
[ 663 1701 1269 1532  576  847 1693 1906  168  150]

True document found at Rank: 10


Q3. Take the top 10 documents retrieved by FAISS in the previous question. Load cross-encoder/ms-marco-MiniLM-L-6-v2. Score the prompt against these 10 documents and sort them by the cross-encoder's score. At what exact rank (1 through 10) does the Cross-Encoder place the true correct document?

In [7]:
# Load the Cross-Encoder
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# Top-10 document indices returned by FAISS
retrieved_indices = indices[0]

# Retrieve the corresponding documents
docs_10 = [kb[i] for i in retrieved_indices]

# Create (question, document) pairs
pairs = [[prompt_150, doc] for doc in docs_10]

# Get Cross-Encoder scores
ce_scores = cross_encoder.predict(pairs)

# Sort documents by score (highest first)
ranking = sorted(
    zip(retrieved_indices, ce_scores),
    key=lambda x: x[1],
    reverse=True
)

print("Cross-Encoder Ranking:")
for rank, (doc_idx, score) in enumerate(ranking, start=1):
    print(f"Rank {rank}: Document {doc_idx} | Score = {score:.4f}")

# Find the rank of the true document (index 150)
true_doc_index = 150

for rank, (doc_idx, score) in enumerate(ranking, start=1):
    if doc_idx == true_doc_index:
        print(f"\nTrue document placed at Rank: {rank}")
        break
else:
    print("\nTrue document was not present in the retrieved top-10 documents.")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Cross-Encoder Ranking:
Rank 1: Document 150 | Score = 4.7585
Rank 2: Document 847 | Score = 4.7526
Rank 3: Document 1693 | Score = 4.7526
Rank 4: Document 1906 | Score = 4.7526
Rank 5: Document 1269 | Score = 4.7375
Rank 6: Document 1532 | Score = 4.7375
Rank 7: Document 168 | Score = 4.7072
Rank 8: Document 576 | Score = 4.6870
Rank 9: Document 663 | Score = 4.6602
Rank 10: Document 1701 | Score = 4.6602

True document placed at Rank: 1


Q4. Retrieve the top k=5 documents for the prompt at row index 42. Concatenate them with a single space between each. Create a string: "Context: [concatenated_docs] Question: [prompt]". Tokenize this string using the bert-base-uncased tokenizer (without truncation). Exactly how many total tokens does this generate?

In [8]:
from transformers import AutoTokenizer

# Get prompt from row 42
row_42 = train.iloc[42]
prompt_42 = str(row_42["prompt"])

# Embed the prompt
query_embedding = model.encode([prompt_42])

# Retrieve top-5 documents
k = 5
distances, indices = index.search(query_embedding, k)

retrieved_docs = [kb[i] for i in indices[0]]

# Concatenate documents with a single space
concatenated_docs = " ".join(retrieved_docs)

# Create the final input string
input_text = f"Context: {concatenated_docs} Question: {prompt_42}"

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Tokenize WITHOUT truncation
tokens = tokenizer(
    input_text,
    truncation=False,
    add_special_tokens=True
)

# Count total tokens
num_tokens = len(tokens["input_ids"])

print(f"Total number of tokens: {num_tokens}")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Total number of tokens: 216


Q5. Retrieve the exact true document for row index 150 from your KB. Create a RAG string: "Context: [true_document] Question: [prompt]". Run the same zero-shot classification from Question 1 on this augmented string. What is the new predicted probability score of the ground-truth correct option? (Round to 3 decimal places).

In [9]:
# Get the true document from the knowledge base
true_document = kb[150]

# Create the RAG prompt
rag_prompt = f"Context: {true_document} Question: {prompt_150}"

# Run zero-shot classification on the augmented prompt
rag_result = zs(
    sequences=rag_prompt,
    candidate_labels=labels_150,
    multi_label=False
)

# Print all scores
print("Scores after adding the true context:\n")
for label, score in zip(rag_result["labels"], rag_result["scores"]):
    print(f"{label[:80]}... -> {score:.6f}")

# Get the score of the correct answer
ground_truth_score = dict(zip(rag_result["labels"], rag_result["scores"]))[ans_150]

print("\nGround Truth Answer:")
print(ans_150)

print(f"\nNew predicted probability score: {ground_truth_score:.3f}")

Scores after adding the true context:

The butterfly effect is the phenomenon that a small change in the initial condit... -> 0.989426
The butterfly effect is the phenomenon that a large change in the initial condit... -> 0.004490
The butterfly effect is the phenomenon that a small change in the initial condit... -> 0.002796
The butterfly effect is the phenomenon that a small change in the initial condit... -> 0.001709
The butterfly effect is the phenomenon that a large change in the initial condit... -> 0.001580

Ground Truth Answer:
The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."

New predicted probability score: 0.989


Q6. What happens if your vector database retrieves the wrong information? Take the prompt for row index 150. Manually force the context to be the document located at KB index 999 (a completely unrelated fact). Run the zero-shot classifier on this "Adversarial RAG" string. What is the probability of the correct option now? (Round to 3 decimal places).

In [10]:
# Get an unrelated document from the knowledge base
wrong_document = kb[999]

# Create the adversarial RAG prompt
adversarial_prompt = f"Context: {wrong_document} Question: {prompt_150}"

# Run zero-shot classification
adv_result = zs(
    sequences=adversarial_prompt,
    candidate_labels=labels_150,
    multi_label=False
)

# Print all scores
print("Scores with adversarial (incorrect) context:\n")
for label, score in zip(adv_result["labels"], adv_result["scores"]):
    print(f"{label[:80]}... -> {score:.6f}")

# Get the probability of the correct answer
correct_score = dict(zip(adv_result["labels"], adv_result["scores"]))[ans_150]

print("\nGround Truth Answer:")
print(ans_150)

print(f"\nProbability of the correct option: {correct_score:.3f}")

Scores with adversarial (incorrect) context:

The butterfly effect is the phenomenon that a small change in the initial condit... -> 0.528949
The butterfly effect is the phenomenon that a large change in the initial condit... -> 0.425296
The butterfly effect is the phenomenon that a small change in the initial condit... -> 0.020378
The butterfly effect is the phenomenon that a large change in the initial condit... -> 0.018688
The butterfly effect is the phenomenon that a small change in the initial condit... -> 0.006689

Ground Truth Answer:
The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."

Probability of the correct option: 0.529


In RAG, a "Hit" occurs if the retrieved context contains the facts needed to answer the question.

Q7. For the first 100 rows of train.csv (indices 0-99), retrieve the top k=5 documents for each prompt. If the exact string of the row's correct option is found inside any of those 5 retrieved documents, it counts as a hit. What is the exact Hit Rate percentage (0 to 100) for these 100 rows? (Round to 1 decimal place).

In [11]:
# Evaluate Hit Rate for the first 100 rows

hits = 0
k = 5

for idx in range(100):
    row = train.iloc[idx]

    # Question
    prompt = str(row["prompt"])

    # Correct answer text
    correct_text = str(row[row["answer"]])

    # Embed the prompt
    query_embedding = model.encode([prompt], show_progress_bar=False)

    # Retrieve top-k documents
    distances, indices = index.search(query_embedding, k)

    retrieved_docs = [kb[i] for i in indices[0]]

    # Check if the exact correct answer string is present
    if any(correct_text in doc for doc in retrieved_docs):
        hits += 1

hit_rate = hits / 100 * 100

print(f"Hits: {hits}/100")
print(f"Hit Rate: {hit_rate:.1f}%")

Hits: 73/100
Hit Rate: 73.0%


Q8. Build a loop that processes the first 20 rows (indices 0 through 19) of train.csv.
For each row, your pipeline must do the following in order:

Retrieve: Embed the prompt and retrieve the top k=5 documents from your FAISS Knowledge Base.

Rerank: Pass the prompt and those 5 documents into the ms-marco-MiniLM-L-6-v2 Cross-Encoder. Select the single document with the highest cross-encoder score.

Augment: Create your RAG string exactly formatted as: "Context: [best_document] Question: [prompt]".

Predict: Pass this augmented string to the facebook/bart-large-mnli zero-shot classifier, using the 5 options (A, B, C, D, E) as your candidate_labels.

Score: Look at the probability scores output by the model. Rank the options from highest probability to lowest. Take the top 3 letters (e.g., ['C', 'A', 'E']) and calculate the MAP@3 for that row.

What is the final average MAP@3 score of this state-of-the-art RAG pipeline across these 20 rows? (Round to 3 decimal places).

In [12]:
# Load Cross-Encoder once
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

k = 5
total_map3 = 0.0

for idx in range(20):

    row = train.iloc[idx]

    prompt = str(row["prompt"])

    # Candidate labels (option texts)
    option_texts = {
        "A": str(row["A"]),
        "B": str(row["B"]),
        "C": str(row["C"]),
        "D": str(row["D"]),
        "E": str(row["E"])
    }

    true_letter = row["answer"]

    # --------------------------------------------------
    # Step 1: Retrieve using FAISS
    # --------------------------------------------------
    query_embedding = model.encode([prompt], show_progress_bar=False)

    distances, indices = index.search(query_embedding, k)

    retrieved_docs = [kb[i] for i in indices[0]]

    # --------------------------------------------------
    # Step 2: Cross-Encoder reranking
    # --------------------------------------------------
    pairs = [[prompt, doc] for doc in retrieved_docs]

    ce_scores = cross_encoder.predict(pairs)

    best_doc = retrieved_docs[np.argmax(ce_scores)]

    # --------------------------------------------------
    # Step 3: Create RAG prompt
    # --------------------------------------------------
    rag_prompt = f"Context: {best_doc} Question: {prompt}"

    # --------------------------------------------------
    # Step 4: Zero-shot prediction
    # --------------------------------------------------
    result = zs(
        sequences=rag_prompt,
        candidate_labels=list(option_texts.values()),
        multi_label=False
    )

    # Convert predicted texts back to option letters
    text_to_letter = {v: k for k, v in option_texts.items()}

    predicted_letters = [
        text_to_letter[label]
        for label in result["labels"]
    ]

    top3 = predicted_letters[:3]

    # --------------------------------------------------
    # Step 5: MAP@3 for one row
    # --------------------------------------------------
    if true_letter in top3:
        rank = top3.index(true_letter) + 1
        ap3 = 1 / rank
    else:
        ap3 = 0

    total_map3 += ap3

average_map3 = total_map3 / 20

print(f"Average MAP@3 over first 20 rows: {average_map3:.3f}")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Average MAP@3 over first 20 rows: 0.975
